## First test of VA in chamber

In [1]:
# Imports
import numpy as np
import time
from datetime import datetime
import os

from mcg_lab_vna_control import connect_vna
from mcg_lab_triaxial_positioner_control import TriaxialPositioner

# Transparent conection: the library knows witch driver to use
vna = connect_vna(brand='keysight')


In [2]:
def time_sleep_sweep_time():
    sweep_time = vna.query_sweep_time(verbose=False)
    time.sleep(6*sweep_time + 1)  # Wait for the sweep to complete

def time_now():
    return f"[{datetime.now().strftime('%H:%M:%S')}]"

In [3]:
########## Parameters ##########
# Triaxial Positioner Port for TCP Server (with Raspi)
CONN_STR = "tcp://192.168.100.2:5000"

# VNA Parameters
ifbw = 10e3
pow_dbm = -15
n_sweeps = 5
n_average = 5

sparam_list = ['S21', 'S31', 'S41', 'S22', 'S33', 'S44']
f_start_list = [2.44996e9, 3.69996e9, 4.94996e9, 7.29984e9, 10.04984e9, 12.79984e9, 14.79984e9] # Segmented f_start list (in Hz)
f_stop_list = [2.55004e9, 3.80004e9, 5.05004e9, 7.70016e9, 10.45016e9, 13.20016e9, 15.20016e9] # Segmented f_stop list (in Hz)
points_list = [279, 279, 279, 557, 557, 557, 557] # Segmented points list

# If Amplifier is used:
# f_start_list = [7.29984e9, 10.04984e9, 12.79984e9, 14.79984e9] # Segmented f_start list (in Hz)
# f_stop_list = [7.70016e9, 10.45016e9, 13.20016e9, 15.20016e9] # Segmented f_stop list (in Hz)
# points_list = [557, 557, 557, 557] # Segmented points list

# Folder name for saving measurement files
folder_name = 'data/VA_chamber_test_1'

if not os.path.exists(folder_name):
    os.makedirs(folder_name, exist_ok=True)

### Basic VNA Config to see Time-Response of the Channel

In [5]:
vna.preset()

vna.configure_multiple_sparams(sparam_list = sparam_list, f_start = 10e9, f_stop = 15e9, points = 5001, ifbw = 10e3, pow_dbm = pow_dbm)

vna.set_timedomain(trace = 1)
vna.set_timedomain(trace = 2)
vna.set_timedomain(trace = 3)
vna.set_timedomain(trace = 4)
vna.set_timedomain(trace = 5)
vna.set_timedomain(trace = 6)

time_sleep_sweep_time()
vna.auto_scale_all()
vna.run_cont()

Measurement time 0.502209354199 seconds
Measurement time 0.502209354199 seconds


In [7]:
vna.set_Xscale(xmin = 160, xmax = 180, mode = 'dist', trace = 1)
vna.set_Xscale(xmin = 160, xmax = 180, mode = 'dist', trace = 2)
vna.set_Xscale(xmin = 160, xmax = 180, mode = 'dist', trace = 3)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 4)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 5)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 6)

time_sleep_sweep_time()
vna.auto_scale_all()
vna.run_cont()

Measurement time 0.502209354199 seconds


### Calibrate the VNA for Segmented Measurements

In [4]:
vna.preset()

vna.configure_multiple_segmented_sparams(sparam_list=sparam_list, f_start_list=f_start_list, f_stop_list=f_stop_list, points_list=points_list, ifbw=ifbw, pow_dbm=pow_dbm)

vna_freqs = vna.get_freq_vector()

time_sleep_sweep_time()
vna.auto_scale_all()
vna.run_cont()

Measurement time 0.304061149172 seconds


### Calibration

#### Port 1:

In [6]:
vna.init_SOL_calibration(port = 1, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 1: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 1"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 1"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 1"


In [ ]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(1)

Measurement of SOL calibrator step 3 completed


In [10]:
# Apply calibration
vna.apply_calibration()

#### Port 2:

In [11]:
vna.init_SOL_calibration(port = 2, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 2: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 2"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 2"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 2"


In [ ]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(1)

Measurement of SOL calibrator step 3 completed


In [15]:
# Apply calibration
vna.apply_calibration()

#### Port 3:

In [ ]:
vna.init_SOL_calibration(port = 3, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

In [ ]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(1)

In [ ]:
# Apply calibration
vna.apply_calibration()

### Measure S21 of Through for back-to-back calibration

In [ ]:
# Measureme S21 of Through
sparam = vna.measure_trace_sparam(trace=1)
vna.auto_scale_all()
np.save(folder_name + '/S21_b2b_cal.npy', sparam)

# Measureme S31 of Through
sparam = vna.measure_trace_sparam(trace=2)
vna.auto_scale_all()
np.save(folder_name + '/S31_b2b_cal.npy', sparam)

# Measureme S41 of Through
sparam = vna.measure_trace_sparam(trace=3)
vna.auto_scale_all()
np.save(folder_name + '/S41_b2b_cal.npy', sparam)

### Positioner test

In [9]:
with TriaxialPositioner(connection_str=CONN_STR) as pos:
    print("Connected to C4 Controller.")
    
    # Configure velocity (optional)
    pos.set_velocity(min_speed=200, max_speed=700, slope=2000)

    pos.home_all_axis()

    pos.move_axes_absolute(z=90, unit='mm')

    pos.move_axes_absolute(z=100, unit='mm')

[C4Controller] Initialized in TCP Client mode -> 192.168.100.2:5000
Connected to C4 Controller.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis X limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Y limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Z limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.


### VA Grid Measurement Loop

In [9]:
########## Parameters ##########
setup_number = 1 # setup number (to have different setup trials)

va_vert_pos = 0 # 0 | 1, idx position in vertical of antenna of the VA  

ue1_deg = 0
ue2_deg = 0
ue3_deg = 0

va_Nh = 30 # VA horizontal (Nh) positions
va_Nv = 14 # VA vertical (Nv) positions
va_deltah = 10.0 # mm of Delta H
va_deltav = 10.0 # mm of Delta V

In [10]:
vna.set_sweeps(sweeps=n_sweeps)
vna.set_average(avg_sweeps=n_average)

freq_vector = vna.get_freq_vector()
freq_points =  len(freq_vector)

trace_list_all = [1, 2, 3, 4, 5, 6] # List of all traces to measure
trace_list_bistatic = [1, 2, 3] # List of bistatic traces to measure

multiple_sweeps_special_measurement = np.zeros((2*va_Nv , n_sweeps, len(trace_list_all), freq_points), dtype=np.complex64)
multiple_sweeps_special_measurement_idx = 0

# The 'with' statement automatically handles connection and safe disconnection
with TriaxialPositioner(connection_str=CONN_STR) as pos:
    print(f"{time_now()} Connected to C4 Controller.")
    
    # Configure velocity (optional)
    pos.set_velocity(min_speed=200, max_speed=700, slope=2000)

    # Move to the home position (0,0,0) before starting the grid scan
    pos.home_all_axis()

    # Establish Absolute Zero
    print(f"{time_now()} Defining current location as Origin (0,0,0)...")
    pos.define_global_coord(x=0.0, y=0.0, z=0.0, unit='mm')

    # Executing a 2D Grid Scan using Generators
    print(f"{time_now()} Starting 2D Grid Scan...")
    grid_iterator = pos.generate_grid_2d(
        axis1='Y', 
        axis2='Z', 
        points1=va_Nh, 
        points2=va_Nv, 
        spacing1=va_deltah, 
        spacing2=va_deltav, 
        unit='mm'
    )
    
    for step in grid_iterator:
        print(f"{time_now()} Arrived at Point {step['point']}/{step['total']} | Y: {step['Y']/10.0}cm, Z: {step['Z']/10.0}cm")

        step_i = step['i']
        step_j = step['j'] + va_vert_pos * va_Nv

        # Initialize row_sparams variable for saving row sparams
        if step_i == 0:
            # Shape: [hor_positions, num_traces, freq_points]
            row_sparams = np.zeros((va_Nh, len(trace_list_all), freq_points), dtype=np.complex64)

        # Measure and collect sparams
        if step_i == 0 or step_i == va_Nh // 2:
            # All measurements both mono and bistatic (S21, S31, S41, S22, S33, S44)
            # 1. Measure with average
            sparams = vna.measure_multiple_traces_sparams(trace_list = trace_list_all)

            # Save the sparams in the row file
            row_sparams[step_i, :, :] = sparams

            # 2. Measure n_sweeps separated without averaging to evaluate the drift during the measurements loop
            print(f"{time_now()} Measuring {n_sweeps} sweeps without averaging")
            vna.set_sweeps(sweeps=1, verbose=False)
            vna.unset_average()
            for sweep in range(n_sweeps):
                sparams = vna.measure_multiple_traces_sparams(trace_list = trace_list_all)
                multiple_sweeps_special_measurement[multiple_sweeps_special_measurement_idx, sweep, :, :] = sparams

            vna.set_sweeps(sweeps=n_sweeps, verbose=False)
            vna.set_average(avg_sweeps=n_average) # Reactivate average            

            multiple_sweeps_special_measurement_idx += 1

        else:
            # Only bistatic measurements (S21, S31, S41)
            sparams = vna.measure_multiple_traces_sparams(trace_list = trace_list_bistatic)

            # Save the sparams in the row file
            row_sparams[step_i, 0:3, :] = sparams

        # Save file for each row
        if step_i == va_Nh - 1: # last horizontal position of the row
            # Current Row Filename
            filename_row = f"{folder_name}/setup_{setup_number}_nv_{step_j}_UE1_{ue1_deg}deg_UE2_{ue2_deg}deg_UE3_{ue3_deg}deg"
            np.save(filename_row, row_sparams)
            time.sleep(1)

    # Save file of multiple sweeps special measurement in specific file
    filename_especial = f"{folder_name}/mult_sweeps_setup_{setup_number}_UE1_{ue1_deg}deg_UE2_{ue2_deg}deg_UE3_{ue3_deg}deg"
    np.save(filename_especial, multiple_sweeps_special_measurement)
    time.sleep(1)

    print(f"{time_now()} Grid scan complete!")

    # Return Home
    print(f"{time_now()} Returning to origin...")
    pos.move_axes_absolute(x=0.0, y=0.0, z=0.0, wait_to_finish=True, unit='mm')

    print(f"{time_now()} Measuring the first VA position (0,0) to evaluate the Drift during the measurements loop")
    # Filename/measurement/saving for extra 0,0 measurement
    filename_extra00 = f"{folder_name}/extra_setup_{setup_number}_nv_0_nh_0_UE1_{ue1_deg}deg_UE2_{ue2_deg}deg_UE3_{ue3_deg}deg"
    sparams = vna.measure_multiple_traces_sparams(trace_list = trace_list_all)
    np.save(filename_extra00, sparams)
    time.sleep(1)

print(f"{time_now()} Routine finished and hardware disconnected safely.")


Measurement time 1.52030574586 seconds
[C4Controller] Initialized in TCP Client mode -> 192.168.100.2:5000
[14:18:16] Connected to C4 Controller.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis X limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Y limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Z limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.
[14:18:23] Defining current location a